# RT-DETR-L Baseline — Chen Split (Tuberculosis6208) — 5-Fold CV

Inline training notebook untuk RT-DETR-L (no wavelet) di dataset AFB Chen split.

**Setup:**
- Dataset zip di Drive: `MyDrive/Tuberculosis6208.zip` (Pascal-VOC format)
- Chen test holdout (fixed): 101 images, `SPLIT_SEED=1050`
- 5-fold CV pada 1164 train+val pool: ~931 train / ~233 val per fold
- Test set 101 images (Chen holdout) sama di semua fold
- Logging: **W&B** project `rtdetr_chen`, group per run name

**Model: RT-DETR-L** (~32M params, HGNetV2 backbone)

**Metrics yg di-track:** mAP50, mAP50-95, mAP@0.9, Precision, Recall, F1

**RT-DETR specifics (from `ultralytics/models/rtdetr/train.py` notes):**
- `F.grid_sample` tidak support `deterministic=True` → diset False
- AMP bisa bikin NaN di bipartite matching → `amp=False`
- Optimizer: AdamW lr=1e-4 (RT-DETR convention, beda dari YOLO SGD)
- Mosaic dimatikan (DETR-class kurang stabil dgn heavy mosaic)

**Runtime estimate:** A100 ~ 40-50 menit per fold → ~4 jam total 5-fold sweep.


## 1. Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Clone repo (branch `dev/wavelet`)


In [ ]:
import os, sys
from pathlib import Path

REPO_DIR = Path('/content/wavelet-yolo12')
BRANCH   = 'dev/wavelet'

if REPO_DIR.exists():
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull --ff-only
else:
    !git clone -b {BRANCH} https://github.com/iswantosan/wavelet-yolo12.git {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('cwd:', os.getcwd())
!git log -1 --oneline


## 3. Install dependencies


In [ ]:
!pip -q install -e . wandb


In [ ]:
import torch, ultralytics
from ultralytics import RTDETR
print('torch       :', torch.__version__, '| cuda:', torch.cuda.is_available())
print('GPU         :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('ultralytics :', ultralytics.__version__)
print('RTDETR OK   :', RTDETR is not None)


## 4. Build Chen split (1024 / 140 / 101, seed=42)

Extract zip → konversi VOC XML → YOLO `.txt` → deterministic shuffle → tulis `data.yaml`. Skip kalau output sudah ada.


In [ ]:
DRIVE_ZIP    = '/content/drive/MyDrive/Tuberculosis6208.zip'
EXTRACT_DIR  = '/content/dataset/raw'
RAW_DIR      = f'{EXTRACT_DIR}/tuberculosis-phonecamera'
SPLIT_DIR    = '/content/tb_chen_split'
CHEN_YAML    = f'{SPLIT_DIR}/data.yaml'

!python scripts/build_chen_split.py \
    --zip "{DRIVE_ZIP}" --extract-dir "{EXTRACT_DIR}" \
    --src "{RAW_DIR}" --out "{SPLIT_DIR}"

!ls -la {SPLIT_DIR} && echo '---' && cat {CHEN_YAML}


## 5. W&B login

Paste API key dari https://wandb.ai/authorize.


In [ ]:
import wandb
wandb.login()


## 6. Config

RT-DETR-L baseline — no wavelet, no NWD. Match epoch/seed/imgsz dgn YOLO baseline untuk apple-to-apple comparison.

**Note:** RT-DETR-L ~32M params (>3× YOLOv12s). Kalau OOM di A100 batch=16, turunin ke batch=8.


In [ ]:
MODEL_CFG    = 'ultralytics/cfg/models/rt-detr/rtdetr-l.yaml'
PRETRAINED   = 'rtdetr-l.pt'      # auto-download from Ultralytics assets
SEED         = 1050
EPOCHS       = 60
IMGSZ        = 640
BATCH        = 16                  # turunin ke 8 kalau OOM
DEVICE       = 0

# K-fold settings
N_FOLDS      = 5
KFOLD_SEED   = 1050
KFOLD_DIR    = '/content/tb_kfold'

WANDB_PROJECT = 'rtdetr_chen'
RUN_PROJECT   = '/content/runs/rtdetr_chen'
RUN_BASE      = f'{Path(MODEL_CFG).stem}_seed{SEED}_{EPOCHS}ep_kf{N_FOLDS}'
GROUP_NAME    = RUN_BASE

print('cfg     :', MODEL_CFG)
print('seed    :', SEED)
print('epochs  :', EPOCHS)
print('n_folds :', N_FOLDS)
print('group   :', GROUP_NAME)


## 7. Build 5-fold splits

Pool 1164 images (Chen train + Chen val), shuffle dgn `KFOLD_SEED=1050`, pecah jadi 5 fold. Tiap fold:
- `train/`: 4 fold lain (~931 imgs)
- `val/`:   1 fold (~233 imgs)
- `test/`:  Chen holdout (101 imgs, sama di semua fold)

Symlink-based — cepat & hemat disk.


In [ ]:
import random, shutil
from pathlib import Path

chen = Path(SPLIT_DIR)
kfold = Path(KFOLD_DIR)

IMG_EXTS = {'.jpg', '.jpeg', '.png'}

def list_imgs(d: Path):
    return sorted([p for p in d.glob('*') if p.suffix.lower() in IMG_EXTS])

def label_for(img: Path) -> Path:
    return img.parent.parent / 'labels' / (img.stem + '.txt')

def sym(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    dst.symlink_to(src.resolve())

train_imgs = list_imgs(chen / 'train' / 'images')
val_imgs   = list_imgs(chen / 'val' / 'images')
test_imgs  = list_imgs(chen / 'test' / 'images')
pool = train_imgs + val_imgs
print(f'Pool train+val : {len(pool)} images')
print(f'Test holdout   : {len(test_imgs)} images (fixed)')

assert len(pool) > 0, f'Empty pool — cek section 4'
assert len(test_imgs) > 0, f'Empty test set — cek {chen}/test/images'

rng = random.Random(KFOLD_SEED)
shuffled = list(pool)
rng.shuffle(shuffled)

fold_size = len(shuffled) // N_FOLDS
folds = [shuffled[i*fold_size:(i+1)*fold_size] for i in range(N_FOLDS)]
for i, img in enumerate(shuffled[N_FOLDS*fold_size:]):
    folds[i].append(img)

if kfold.exists():
    shutil.rmtree(kfold)

FOLD_YAMLS = []
for k in range(N_FOLDS):
    val_k   = folds[k]
    val_set = set(val_k)
    train_k = [im for f in folds[:k] + folds[k+1:] for im in f]

    fdir = kfold / f'fold{k}'
    for split_name, imgs in [('train', train_k), ('val', val_k), ('test', test_imgs)]:
        for img in imgs:
            sym(img, fdir / split_name / 'images' / img.name)
            lbl = label_for(img)
            if lbl.exists():
                sym(lbl, fdir / split_name / 'labels' / lbl.name)

    # Per-fold data.yaml
    yaml_txt = (
        f'path: {fdir.resolve()}\n'
        f'train: train/images\n'
        f'val: val/images\n'
        f'test: test/images\n'
        f'nc: 1\n'
        f"names: ['AFB']\n"
    )
    yaml_path = fdir / 'data.yaml'
    yaml_path.write_text(yaml_txt)
    FOLD_YAMLS.append(str(yaml_path))
    print(f'fold{k}: train={len(train_k)} val={len(val_k)} test={len(test_imgs)} → {yaml_path}')

print(f'\nAll {N_FOLDS} fold yamls ready.')


## 8. Seed + Ultralytics callback setup

Disable built-in W&B callback → log manual per fold.

**RT-DETR note:** TIDAK pakai SDP math kernel override (RT-DETR pakai `F.grid_sample`, bukan SDPA).


In [ ]:
import os, gc, random, numpy as np, torch

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

from ultralytics.utils import SETTINGS
SETTINGS.update({'wandb': False})
print('Seed + Ultralytics W&B callback disabled.')


## 9. Helper functions (eval + W&B csv-replay)

**`evaluate()`** menghitung:
- mAP50, mAP50-95 (built-in)
- mAP@0.9 (ekstrak dari `box.all_ap` index ke-8)
- Precision, Recall (mean over classes)
- **F1** = 2·P·R / (P+R) — derived

**`log_csv_to_wandb()`** replay `results.csv` per epoch. RT-DETR pakai kolom loss `giou_loss`, `cls_loss`, `l1_loss` (beda dari YOLO `box_loss`, `cls_loss`, `dfl_loss`).


In [ ]:
import pandas as pd

EVAL_KEYS = ('mAP50', 'mAP50-95', 'mAP@0.9', 'precision', 'recall', 'F1')

def evaluate(model, data_yaml, split):
    """Run model.val() on given split, return metrics dict."""
    eva = model.val(data=data_yaml, split=split, imgsz=IMGSZ, device=DEVICE, verbose=False)
    p = float(np.mean(np.atleast_1d(eva.box.p)))
    r = float(np.mean(np.atleast_1d(eva.box.r)))
    f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
    out = {
        'mAP50':     float(eva.box.map50),
        'mAP50-95':  float(eva.box.map),
        'precision': p,
        'recall':    r,
        'F1':        f1,
        'mAP@0.9':   float('nan'),
    }
    try:
        ap_all = eva.box.all_ap
        if ap_all is not None and len(ap_all):
            ap = ap_all.mean(axis=0) if (hasattr(ap_all, 'ndim') and ap_all.ndim == 2) else ap_all
            if len(ap) >= 9:
                out['mAP@0.9'] = float(ap[8])
    except Exception as e:
        print(f'  (mAP@0.9 extract failed: {e})')
    return out


def log_csv_to_wandb(run, csv_path):
    """Replay results.csv per epoch into active W&B run."""
    wandb.define_metric('epoch')
    for k in [
        'train/giou_loss', 'train/cls_loss', 'train/l1_loss',
        'val/giou_loss', 'val/cls_loss', 'val/l1_loss',
        'val/mAP50', 'val/mAP50-95', 'val/precision', 'val/recall', 'lr/pg0',
    ]:
        wandb.define_metric(k, step_metric='epoch')

    if not Path(csv_path).exists():
        print(f'  results.csv missing: {csv_path}')
        return
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    for _, row in df.iterrows():
        log_data = {'epoch': int(row.get('epoch', 0))}
        for col in df.columns:
            if col == 'epoch':
                continue
            try:
                v = float(row[col])
                if not np.isnan(v):
                    log_data[col] = v
            except (ValueError, TypeError):
                pass
        run.log(log_data)


## 10. K-fold training loop

Tiap fold = satu W&B run dengan `group=GROUP_NAME`. Per fold:
1. Train (`EPOCHS` epoch) RT-DETR-L dgn `data.yaml` fold tersebut
2. Log per-epoch curves dari `results.csv`
3. Evaluate `best.pt` di **val** (fold-specific) dan **test** (Chen holdout)
4. Log summary metrics ke W&B, cleanup GPU/RAM

**Hyperparam notes (vs YOLO baseline):**
- `optimizer='AdamW'` + `lr0=1e-4` (RT-DETR convention)
- `amp=False` (avoid NaN in bipartite matching)
- `deterministic=False` (grid_sample limitation)
- `mosaic=0` (DETR doesn't like aggressive mosaic)


In [ ]:
import time
from ultralytics import RTDETR

all_results = []

for k, fold_yaml in enumerate(FOLD_YAMLS):
    run_name = f'{RUN_BASE}_fold{k}'
    print(f'\n{"="*70}\n  FOLD {k}/{N_FOLDS-1}  ->  {run_name}\n{"="*70}')

    run = wandb.init(
        project=WANDB_PROJECT,
        group=GROUP_NAME,
        name=run_name,
        reinit=True,
        job_type='train',
        config=dict(
            fold=k, n_folds=N_FOLDS, kfold_seed=KFOLD_SEED,
            model_cfg=MODEL_CFG, data_yaml=fold_yaml, pretrained=PRETRAINED,
            seed=SEED, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
            optimizer='AdamW', lr0=1e-4, weight_decay=1e-4, cos_lr=True,
            amp=False, deterministic=False, mosaic=0,
            split=f'kfold{N_FOLDS}_chen_holdout',
        ),
        tags=[Path(MODEL_CFG).stem, f'seed{SEED}', f'kfold{N_FOLDS}', f'fold{k}'],
    )
    print('  W&B run:', run.url)

    # ---- Train ----
    model = RTDETR(MODEL_CFG)
    try:
        model.load(PRETRAINED)
        print(f'  Loaded pretrained: {PRETRAINED}')
    except Exception as e:
        print(f'  [warn] could not load pretrained: {e}')

    t0 = time.time()
    results = model.train(
        data=fold_yaml,
        epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
        optimizer='AdamW', lr0=1e-4, weight_decay=1e-4, cos_lr=True, patience=0,
        amp=False, deterministic=False, seed=SEED, workers=8,
        # Mild augmentation (RT-DETR-friendly)
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=0.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0,
        flipud=0.0, fliplr=0.5,
        mosaic=0.0, mixup=0.0, copy_paste=0.0,
        project=RUN_PROJECT, name=run_name, exist_ok=True,
        verbose=True,
    )
    train_min = (time.time() - t0) / 60.0
    print(f'  Training done in {train_min:.1f} min')

    save_dir = Path(results.save_dir if hasattr(results, 'save_dir') else f'{RUN_PROJECT}/{run_name}')
    csv_path = save_dir / 'results.csv'
    log_csv_to_wandb(run, csv_path)

    # ---- Eval best.pt on val & test ----
    best_pt = save_dir / 'weights' / 'best.pt'
    print(f'  Loading best: {best_pt}')
    best_model = RTDETR(str(best_pt))

    val_metrics  = evaluate(best_model, fold_yaml, 'val')
    test_metrics = evaluate(best_model, fold_yaml, 'test')

    print(f'  val  : {val_metrics}')
    print(f'  test : {test_metrics}')

    # Log summary to W&B
    summary = {'fold': k, 'train_min': train_min}
    for m, v in val_metrics.items():
        summary[f'val/{m}'] = v
    for m, v in test_metrics.items():
        summary[f'test/{m}'] = v
    run.log(summary)
    run.summary.update(summary)

    all_results.append({
        'fold': k,
        'val': val_metrics,
        'test': test_metrics,
        'train_min': train_min,
        'save_dir': str(save_dir),
    })

    run.finish()
    del model, best_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'\n{"="*70}\n  ALL {N_FOLDS} FOLDS DONE\n{"="*70}')


## 11. Cross-fold aggregation (mean ± std)

Log summary run `<RUN_BASE>_SUMMARY` ke W&B berisi mean/std semua metrik val & test (mAP50, mAP50-95, mAP@0.9, P, R, F1).


In [ ]:
import math, statistics

def _valid(xs):
    return [x for x in xs if x is not None and not (isinstance(x, float) and math.isnan(x))]

agg = {}
print(f'\n=== {N_FOLDS}-FOLD CV SUMMARY ({RUN_BASE}) ===\n')
print(f"{'Split/Metric':<22}{'Mean':>10}{'Std':>10}{'Min':>10}{'Max':>10}")
print('-' * 62)
for split in ('val', 'test'):
    for m in EVAL_KEYS:
        vals = _valid([f[split].get(m) for f in all_results])
        if not vals:
            continue
        mean = statistics.mean(vals)
        std  = statistics.stdev(vals) if len(vals) > 1 else 0.0
        agg[f'{split}/{m}/mean'] = mean
        agg[f'{split}/{m}/std']  = std
        agg[f'{split}/{m}/min']  = min(vals)
        agg[f'{split}/{m}/max']  = max(vals)
        print(f'{split}/{m:<16}{mean:>10.4f}{std:>10.4f}{min(vals):>10.4f}{max(vals):>10.4f}')

train_mins = _valid([f['train_min'] for f in all_results])
agg['train/time_min/mean'] = statistics.mean(train_mins) if train_mins else 0.0
agg['train/time_min/sum']  = sum(train_mins) if train_mins else 0.0
print('-' * 62)
print(f"train_min (avg/total)  {agg['train/time_min/mean']:>10.1f}{'':>10}{'':>10}{agg['train/time_min/sum']:>10.1f}")

summary_run = wandb.init(
    project=WANDB_PROJECT,
    group=GROUP_NAME,
    name=f'{RUN_BASE}_SUMMARY',
    reinit=True,
    job_type='summary',
    config=dict(
        model_cfg=MODEL_CFG, n_folds=N_FOLDS, kfold_seed=KFOLD_SEED,
        seed=SEED, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    ),
    tags=[Path(MODEL_CFG).stem, f'seed{SEED}', f'kfold{N_FOLDS}', 'summary'],
)
summary_run.log(agg)
summary_run.summary.update(agg)
summary_run.finish()
print(f'\nSummary run logged to W&B project {WANDB_PROJECT}, group {GROUP_NAME}.')


## 12. (Opsional) Quick predict sample dari fold-0 best.pt


In [ ]:
from ultralytics import RTDETR

if all_results:
    best_pt = Path(all_results[0]['save_dir']) / 'weights' / 'best.pt'
    test_dir = Path(KFOLD_DIR) / 'fold0' / 'test' / 'images'
    pred_model = RTDETR(str(best_pt))
    preds = pred_model.predict(
        source=str(test_dir),
        save=True, imgsz=IMGSZ, conf=0.25, device=DEVICE,
    )
    print('Predictions saved to:', preds[0].save_dir if preds else None)
else:
    print('No fold results to predict from.')
